# Getting the weather data
- getting data from open-meteo.com

In [1]:
import requests #needed for the api call

In [2]:
#use api here and requests to get data from the api
url = "https://api.open-meteo.com/v1/forecast?latitude=45.5234&longitude=-122.6762&hourly=temperature_2m&forecast_days=3"
resp = requests.get(url, timeout = 30)
print(resp.status_code)

200


get code 200, things are good!

In [3]:
data = resp.json() #get the data in json format
data.keys() #get the keys of the data

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])

In [4]:
print(data["hourly"].keys())#get the keys of the hourly data

data["hourly"]["temperature_2m"][:5] #get the first 5 temperature values

dict_keys(['time', 'temperature_2m'])


[22.1, 20.5, 18.8, 17.5, 16.9]

In [5]:
#get params to shorten url and make it easier to read
p1 = data['latitude']
print(p1)

p2 = data['longitude']
print(p2)

start_date = data['hourly']['time'][0]
print(start_date)

end_date = data['hourly']['time'][-1]
print(end_date)

45.528744
-122.696236
2026-10-07T00:00
2026-10-09T23:00


Don't match the url right now but will need this idea for a function, copy from above url

In [6]:
BASE_URL = "https://api.open-meteo.com/v1/forecast"
lat = 45.5234
lon = -122.6762
hourly = "temperature_2m"
forecast_days = 3
params = {
    "latitude": lat,
    "longitude": lon,
    "hourly": hourly,
    "forecast_days": forecast_days
    }

In [7]:
#test the params by making a request to the api
resp = requests.get(BASE_URL, params=params, timeout=30)
print(resp.status_code)

200


IT worked here as well, change a variable to see if it still works. Try Vancouver WA

In [11]:
lat = 45.6387 ; lon = -122.6615 ;
params = {
    "latitude": lat,
    "longitude": lon,
    "hourly": hourly,
    "forecast_days": forecast_days
    }

resp = requests.get(BASE_URL, params=params, timeout=30)
print(resp.status_code)

200


And changing still gets 200, no check of different values but call was not shot down so moving forward. I also am using the forecast data, will need the past data as well which uses a different base. 

In [12]:
import pandas as pd #need for dataframe

In [13]:
df = pd.DataFrame(data['hourly'])
df.head() #get the first 5 rows of the dataframe

,time,temperature_2m
0,2026-10-07T00:00,22.1
1,2026-10-07T01:00,20.5
2,2026-10-07T02:00,18.8
3,2026-10-07T03:00,17.5
4,2026-10-07T04:00,16.9


Got a dataframe with time and temperature. 
Time units are going to need to be adjusted


In [14]:
df['time'] = pd.to_datetime(df['time'], utc=True) #convert time to datetime format

In [15]:
df.dtypes #check the data types 

time              datetime64[us, UTC]
temperature_2m                float64
dtype: object

In [16]:
#use the archived data from meteo

## Turn this into a function that can be used to pull data
    - The above was done to get the calls and make sure I was able to get it correct, the function will now make it much more usable

In [17]:
import pandas as pd
import requests

BASE_URL = "https://api.open-meteo.com/v1/forecast"
HOURLY_VARS = ['temperature_2m'] #this can change based on what I want, right now using above set to temperature_2m

def fetch_weather(lat: float, lon: float,
                  forecast_days: int = 3, hourly_vars: list = None):

    #check if given or set to global default
    if hourly_vars is None:
        hourly_vars = HOURLY_VARS

    #build params for the request
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": ",".join(hourly_vars), #so it can be a list of variables
        "forecast_days": forecast_days
    }

    #make the request to the API
    resp = requests.get(BASE_URL, params=params, timeout=30)

    #check if the request was successful
    # if resp.status_code != 200:
    #     raise Exception(f"Request failed with status code {resp.status_code}")
    #request has its own error handling
    resp.raise_for_status()

    #make dataframe from the json data
    data = resp.json()
    df = pd.DataFrame(data['hourly'])

    #update time
    df['time'] = pd.to_datetime(df['time'], utc=True)
    return df
    

In [18]:
#test function
lat = 45.6387 ; lon = -122.6615 ;
df = fetch_weather(lat, lon)

In [19]:
df.head()

,time,temperature_2m
0,2026-10-07 00:00:00+00:00,22.8
1,2026-10-07 01:00:00+00:00,21.0
2,2026-10-07 02:00:00+00:00,19.1
3,2026-10-07 03:00:00+00:00,17.8
4,2026-10-07 04:00:00+00:00,16.8


This check out, this function will need updating as I get more of what to expect but the skeleton base is there. Adding to script so edits can live there as this goes. 